In [9]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.cluster import KMeans
import warnings

from src.features import engineer_features
warnings.filterwarnings('ignore')

In [10]:
print("Loading data and engineering autoregressive lags...")
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train['geohash'] = train['geohash'].fillna('Missing')
test['geohash'] = test['geohash'].fillna('Missing')

# ─── GRANDMASTER UPGRADE 1: EXACT HISTORICAL LAGS ───
# We shift the training data forward by 1 day (24h) and 7 days (1 week)
# to give the model the exact demand from the recent past.

# 1. Create lookup dictionaries from the training data
lag_24h_df = train.copy()
lag_24h_df['target_day'] = lag_24h_df['day'] + 1  # Shift forward 1 day
lag_24h_map = lag_24h_df.set_index(['target_day', 'geohash', 'timestamp'])['demand'].to_dict()

lag_7d_df = train.copy()
lag_7d_df['target_day'] = lag_7d_df['day'] + 7    # Shift forward 7 days
lag_7d_map = lag_7d_df.set_index(['target_day', 'geohash', 'timestamp'])['demand'].to_dict()

# Combine datasets for unified engineering
test['demand'] = -1 
df = pd.concat([train, test], ignore_index=True)

# 2. Map the lags to both train and test sets
# If test is Day 49, it will look up Day 48 and Day 42 from the train data safely!
df['lag_24h'] = df.set_index(['day', 'geohash', 'timestamp']).index.map(lag_24h_map)
df['lag_7d'] = df.set_index(['day', 'geohash', 'timestamp']).index.map(lag_7d_map)

Loading data and engineering autoregressive lags...


In [11]:
print("Applying spatial-temporal features and K-Means zones...")
df_engineered = engineer_features(df)

# ─── GRANDMASTER UPGRADE 2: K-MEANS SPATIAL ZONES ───
# Group coordinates into 50 geographic neighborhoods
coords = df_engineered[['latitude', 'longitude']].dropna()
# Using n_init='auto' to prevent warnings and speed up clustering
kmeans = KMeans(n_clusters=50, random_state=42, n_init='auto') 
df_engineered['spatial_zone'] = -1
df_engineered.loc[coords.index, 'spatial_zone'] = kmeans.fit_predict(coords)

# --- Impute Missing Lags Safely ---
# For rows where 24h or 7d lag doesn't exist, we fill with 0 to represent baseline
df_engineered['lag_24h'] = df_engineered['lag_24h'].fillna(0)
df_engineered['lag_7d'] = df_engineered['lag_7d'].fillna(0)

# Split back to Train/Test
train_df = df_engineered[df_engineered['demand'] != -1].copy()
test_df = df_engineered[df_engineered['demand'] == -1].copy()

X = train_df.drop(columns=['Index', 'demand'])
y = train_df['demand']
X_test = test_df.drop(columns=['Index', 'demand'])

# Add spatial_zone to our categorical list so CatBoost/LightGBM process it optimally
cat_features = ['geohash', 'RoadType', 'Weather', 'spatial_zone']

Applying spatial-temporal features and K-Means zones...


In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

print("Initiating Leak-Proof Out-of-Fold Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
    X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx].copy()
    X_test_fold = X_test.copy()
    
    # --- THE UPGRADE: Strict Out-Of-Fold Target Encoding ---
    # Calculates explicit historical traffic baselines without leaking the answers
    encoder = TargetEncoder(target_type='continuous', random_state=42)
    
    X_train['geo_historical_demand'] = encoder.fit_transform(X_train[['geohash']], y_train)
    X_val['geo_historical_demand'] = encoder.transform(X_val[['geohash']])
    X_test_fold['geo_historical_demand'] = encoder.transform(X_test_fold[['geohash']])
    
    # --- Format for LightGBM ---
    X_train_lgb, X_val_lgb, X_test_lgb = X_train.copy(), X_val.copy(), X_test_fold.copy()
    for col in cat_features:
        X_train_lgb[col] = X_train_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    lgb = LGBMRegressor(n_estimators=2500, learning_rate=0.02, random_state=42, n_jobs=-1)
    lgb.fit(X_train_lgb, y_train, eval_set=[(X_val_lgb, y_val)], 
            callbacks=[early_stopping(stopping_rounds=100), log_evaluation(period=0)])
    lgb_test_preds += lgb.predict(X_test_lgb) / kf.n_splits
    
    # --- Format for CatBoost ---
    X_train_cat, X_val_cat, X_test_cat = X_train.copy(), X_val.copy(), X_test_fold.copy()
    for col in cat_features:
        X_train_cat[col] = X_train_cat[col].astype(str)
        X_val_cat[col] = X_val_cat[col].astype(str)
        X_test_cat[col] = X_test_cat[col].astype(str)
        
    cat = CatBoostRegressor(iterations=2500, learning_rate=0.03, cat_features=cat_features, random_seed=42, verbose=0)
    cat.fit(X_train_cat, y_train, eval_set=(X_val_cat, y_val), early_stopping_rounds=100)
    cat_test_preds += cat.predict(X_test_cat) / kf.n_splits
    
    print(f"Fold {fold+1} Completed.")

# Robust Blended Predictions
final_predictions = (lgb_test_preds * 0.5) + (cat_test_preds * 0.5)

# Safety clip prevents impossible negative demands
final_predictions = np.clip(final_predictions, a_min=0.0, a_max=None)

submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully! Ready for upload.")

Initiating Leak-Proof Out-of-Fold Ensemble...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005856 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2112
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 16
[LightGBM] [Info] Start training from score 0.093784
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2500]	valid_0's l2: 0.000802567
Fold 1 Completed.
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bi